# Mellanox 首次纳入 Data Center 收入：原始 SEC 来源定义修复

仅读 2020-08-19 原件一次，结合已保存对照；四个同比事件标记定义未知，不读取 SOXX 结果。


In [1]:
import sys,json,hashlib,re,tarfile,tempfile,subprocess,shlex
from pathlib import Path
from bs4 import BeautifulSoup
root=Path('/Users/rziiiii/dev/workspace');sys.path.insert(0,str(root))
from src.data.nas_market_data import _settings,nas_api_call
card=root/'research/stage1_nvda_datacenter_mellanox_semantics_v2_20261023';plan=json.loads((card/'plan.json').read_text());sha=lambda p:hashlib.sha256(Path(p).read_bytes()).hexdigest()
assert plan['frozen'] and all(plan['budget'][k]==0 for k in ('new_HTTP_GETs','market_outcome_reads','model_fits','native_accounts','new_Qlib_Recorders'))
assert sha(root/plan['predecessor_failure'])==plan['predecessor_failure_sha256'] and sha(root/plan['source_result'])==plan['source_result_sha256']
p=root/plan['exact_original']['path'];payload=p.read_bytes();digest=hashlib.sha256(payload).hexdigest();assert digest==plan['exact_original']['raw_sha256']
full=payload.decode('utf-8','replace');docs=[m.group(1) for m in re.finditer(r'<DOCUMENT>(.*?)</DOCUMENT>',full,re.I|re.S)];texts=[]
for doc in docs:
 if not re.search(r'<TYPE>\s*EX-99\.[12](?:\s|<)',doc,re.I):continue
 m=re.search(r'<TEXT>(.*?)</TEXT>',doc,re.I|re.S)
 if m:texts.append(' '.join(BeautifulSoup(m.group(1),'html.parser').stripped_strings))
text=re.sub(r'\s+',' ',' '.join(texts));closed=bool(re.search(r'(?i)(?:closed|completed).{0,120}Mellanox.{0,100}April 27, 2020|Mellanox.{0,180}April 27, 2020',text));included=bool(re.search(r'(?i)Data Center revenue.{0,90}includes Mellanox|Data Center.{0,180}Mellanox contributed|Mellanox.{0,180}Data Center revenue',text))
assert len(texts)==2 and closed and included,(len(texts),closed,included)
parent=json.loads((root/plan['source_result']).read_text());events={e['accession']:e for e in parent['events']};mask=[]
for frozen in plan['frozen_four_accessions']:
 e=events[frozen['accession']];assert all(e[k]==v for k,v in frozen.items()) and e['status']=='numeric_same_release_table'
 mask.append({'accession':e['accession'],'filing_date':e['filing_date'],'first_conservative_Qlib_signal_date':e['first_conservative_Qlib_signal_date'],'raw_sha256':e['raw_sha256'],'comparability':'unknown_Mellanox_current_prior_year_asymmetry'})
post=events[plan['first_comparable_after_one_year']['accession']];assert all(post[k]==v for k,v in plan['first_comparable_after_one_year'].items())
pre=json.loads((root/plan['predecessor_failure']).read_text());assert pre['source_control_observations'][0]['Q1_outlook_next_quarter_Mellanox'] and not pre['source_control_observations'][1]['Q2_initial_inclusion']
result={'decision':'original_SEC_Mellanox_four_YoY_events_incomparable_source_mask','plan_sha256':sha(card/'plan.json'),'new_raw_original_reads':1,'first_inclusion_original':{'accession':plan['exact_original']['accession'],'raw_sha256':digest,'EX99_document_text_count':len(texts),'Mellanox_close_April27_2020_in_original':closed,'Data_Center_includes_Mellanox_in_original':included,'selected_text_sha256':hashlib.sha256(text.encode()).hexdigest()},'predecessor_two_read_failure_preserved':True,'masked_events':mask,'first_after_one_year_original_reference':plan['first_comparable_after_one_year'],'numeric_values_changed':False,'other_definition_changes_qualified_absent':False,'issuer_first_publication_qualified':False,'market_outcome_reads':0,'model_fits':0,'native_accounts':0}
(card/'result.json').write_text(json.dumps(result,indent=2)+'\n')
settings=_settings(root);files=[card/'plan.json',card/'result.json']
with tempfile.TemporaryDirectory(prefix='nvda-mellanox-v2-') as tmp:
 a=Path(tmp)/'source.tar.gz'
 with tarfile.open(a,'w:gz') as out:
  for f in files:out.add(f,arcname=str(f.relative_to(root)))
 h=hashlib.sha256(a.read_bytes()).hexdigest();response=nas_api_call(origin=settings['origin'],token=settings['token'],method='PUT',path='/api/research/artifacts/'+h,upload=a,executor_id='soxl-minute-research');rec=response['artifact'];assert rec['sha256']==h and rec['size_bytes']==a.stat().st_size
 remote=settings['artifact_root'].rstrip('/')+'/'+h[:2]+'/'+h[2:4]+'/'+h;restore=Path(tmp)/'restore.tar.gz'
 with restore.open('wb') as out:proc=subprocess.run(['ssh','-p',str(settings['ssh']['port']),'-o','BatchMode=yes',settings['ssh']['host'],'cat '+shlex.quote(remote)],stdout=out,stderr=subprocess.PIPE,timeout=90)
 assert proc.returncode==0 and hashlib.sha256(restore.read_bytes()).hexdigest()==h
 with tarfile.open(restore,'r:gz') as z:
  names={m.name:m for m in z.getmembers() if m.isfile()};assert set(names)=={str(f.relative_to(root)) for f in files}
  for f in files:assert hashlib.sha256(z.extractfile(names[str(f.relative_to(root))]).read()).hexdigest()==sha(f)
receipt={'status':'semantics_source_NAS_scoped_restore_verified','sha256':h,'size_bytes':rec['size_bytes'],'nas_path':remote,'members':{str(f.relative_to(root)):sha(f) for f in files},'verified':True};(card/'nas_source_archive.json').write_text(json.dumps(receipt,indent=2)+'\n')
summary={'experiment':plan['experiment'],'decision':result['decision'],'objective_achieved':False,'plan_sha256':result['plan_sha256'],'result':'result.json','source_archive':'nas_source_archive.json','new_raw_original_reads':1,'model_fits':0,'native_accounts':0,'artifact_policy':plan['artifact_policy'],'limitations':['Four reported YoY events are source-incomparable due to Mellanox current/prior-year asymmetry.','The first after-one-year event is a comparability boundary only, not proof all later platform definitions are stable.','No SOXX market outcome or trading account.']}
(card/'summary.json').write_text(json.dumps(summary,indent=2)+'\n')
print(json.dumps({'closed':closed,'included':included,'masked':[e['accession'] for e in mask],'nas_sha256':h},indent=2))


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


{
  "closed": true,
  "included": true,
  "masked": [
    "0001045810-20-000145",
    "0001045810-20-000187",
    "0001045810-21-000007",
    "0001045810-21-000063"
  ],
  "nas_sha256": "06b7486cf4a6297a8db0f67233627b5866b7359b75bd5a64db145c22d11e532e"
}


# Mellanox 首次纳入 NVIDIA Data Center 收入：同比可比性边界

第一版把 2021-08-18 误当作“首次纳入”对照，在两次原件读取后被严格断言拦下并保留失败收据。本版另行冻结，只读 **2020-08-19** 的原始 SEC 盈利公告一次；原件明确记录 Mellanox 于 **2020-04-27** 完成收购，且当季 Data Center 收入包含 Mellanox。2020-05-21 的已保存对照在前一季对下一季包含收购的指引中说明变化；没有为本版重读它。

因此把四个原件同比事件——2020-08-19、2020-11-18、2021-02-24、2021-05-26——标为**当前季含收购、上年同期不含收购的结构不可比**。原报告金额和同比原值完整保留，但在后续信息特征中应保持未知，不把收购增量解释成同口径 AI 需求增长。2021-08-18 是首个当前季与上年同季均在收购之后的来源边界；这不证明此后的全部产品/平台定义恒定。

原始 SEC 字节与本卡结果已在 Yanxuan NAS 按 SHA-256 归档并限定恢复。本版 1 次原件读取、0 新 HTTP、0 SOXX 市场收益、0 拟合、0 账户。决定：`original_SEC_Mellanox_four_YoY_events_incomparable_source_mask`。该来源语义掩码须在单独冻结的 Qlib 信息检验前应用；SEC 接收后下一交易日仍只是保守可用代理，非发行人首次公开时刻证明。
